# Análisis de video de fútbol — JPE AI Solutions

**Antes de ejecutar nada:** activa la GPU en
*Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*.

Luego ejecuta las celdas en orden (botón ▶ de cada una).

In [ ]:
# 1. Comprobar que hay GPU disponible
!nvidia-smi

## 2. Subir el código del proyecto

Al ejecutar la celda siguiente aparecerá un botón **Elegir archivos**:
selecciona `analisis-video-src.zip` (está en la carpeta del proyecto,
visible desde *Archivos de Linux* en tu Chromebook).

In [ ]:
from google.colab import files
uploaded = files.upload()
!unzip -o -q analisis-video-src.zip -d analisis-video
%cd analisis-video

In [ ]:
# 3. Instalar dependencias (torch con CUDA ya viene instalado en Colab)
!pip install -q ultralytics supervision trackers easyocr ffmpeg-python

## 4. El video del partido

Sube antes el video a tu **Google Drive** (desde el Chromebook:
app Archivos → arrastrar a Google Drive). Ajusta la ruta `VIDEO`
de la celda siguiente al nombre real de tu archivo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

VIDEO = '/content/drive/MyDrive/partido.mp4'  # <-- AJUSTA ESTA RUTA

import os
assert os.path.exists(VIDEO), f'No se encuentra el video en: {VIDEO}'
print('Video encontrado:', VIDEO)

## 5. Ejecutar el análisis

Para un partido completo tarda ~1 hora con GPU T4.

Opciones útiles:
- `--stride 2` procesa 1 de cada 2 frames (el doble de rápido, apenas pierde precisión)
- `--no-annotated-video` no genera el video anotado (mucho más rápido y ligero)
- `--start 0 --end 120` procesa solo los primeros 2 minutos (para probar primero)
- `--ocr` activa la lectura del marcador en pantalla (si el video lo muestra)
- `--calibration cancha.json` para distancias en metros y eventos de gol/córner
  (ver formato en `src/analisis_video/pitch.py`)

In [ ]:
# Prueba rápida recomendada: solo los primeros 2 minutos
!python scripts/run_pipeline.py --video "{VIDEO}" --device 0 --stride 2 --end 120

In [ ]:
# Partido completo (ejecutar cuando la prueba de arriba se vea bien)
!python scripts/run_pipeline.py --video "{VIDEO}" --device 0 --stride 2 --ocr

In [ ]:
# 6. Guardar los resultados en tu Google Drive
!zip -r -q resultados_analisis.zip outputs
!cp resultados_analisis.zip /content/drive/MyDrive/
print('Resultados guardados en Google Drive: resultados_analisis.zip')